# CreditPilot — Financial Stability Index (FSI) & Risk Engine

## Overview
This notebook implements the Financial Stability Index (FSI) and Risk Engine, which extend the machine learning model into a complete decision-support system. 

While the ML model predicts default probability, the FSI evaluates overall financial strength, and the Risk Engine combines both to make a final credit decision.

---

## Objectives

- Build Financial Stability Index (FSI)  
- Classify users into risk categories  
- Combine ML prediction + FSI  
- Create final loan decision logic  

---

## Key Components

### 1. Financial Stability Index (FSI)
A composite score (0–100) based on:
- Income stability  
- Credit score  
- Debt-to-Income ratio  
- EMI burden  
- Credit utilization  

---

### 2. Risk Engine
Combines:
- ML prediction probability  
- FSI score  
- Behavioral signals  

---

### 3. Final Decision
- APPROVED  
- CONDITIONAL  
- REJECTED  

---

##  Next Step

Stress Simulation Engine

In [1]:
import pandas as pd
import numpy as np
import joblib

In [8]:
model = joblib.load("../models/final_model.pkl")

raw_df = pd.read_csv("../data/raw/creditpilot_dataset.csv")
processed_df = pd.read_csv("../data/processed/final_dataset.csv")



## Feature Prep

In [9]:
X = processed_df.drop("loan_status", axis=1)

## Get Model Probability

In [10]:
prob = model.predict_proba(X)[:, 1]

raw_df["default_probability"] = prob

## Financial Stability Index

In [11]:
credit_norm = (raw_df["credit_score"] - 300) / (850 - 300)

dti_score = (1 - raw_df["debt_to_income_ratio"])
emi_score = (1 - raw_df["emi_to_income_ratio"])
util_score = (1 - raw_df["credit_utilization_ratio"])

In [12]:
raw_df["fsi"] = (
    0.25 * df["income_stability_score"]/100 +
    0.20 * credit_norm +
    0.20 * dti_score +
    0.15 * emi_score +
    0.20 * util_score
) * 100

## Risk Category

In [13]:
def get_risk(prob, fsi):
    if prob > 0.7 or fsi < 50:
        return "High Risk"
    elif prob > 0.4 or fsi < 65:
        return "Medium Risk"
    else:
        return "Low Risk"

raw_df["risk_category"] = [
    get_risk(p, f) for p, f in zip(raw_df["default_probability"], raw_df["fsi"])
]

## Final Decision Engine

In [14]:
def final_decision(prob, fsi):
    if prob < 0.4 and fsi > 70:
        return "APPROVED"
    elif prob < 0.6 and fsi > 55:
        return "CONDITIONAL"
    else:
        return "REJECTED"

raw_df["final_decision"] = [
    final_decision(p, f) for p, f in zip(raw_df["default_probability"], raw_df["fsi"])
]

## Check Output

In [15]:
raw_df[["default_probability", "fsi", "risk_category", "final_decision"]].head()

,default_probability,fsi,risk_category,final_decision
0,0.075247,62.032943,Medium Risk,CONDITIONAL
1,0.007892,45.660147,High Risk,REJECTED
2,0.996996,55.546176,High Risk,REJECTED
3,0.954731,35.538143,High Risk,REJECTED
4,0.999979,21.552588,High Risk,REJECTED


## Distribution

In [16]:
raw_df["risk_category"].value_counts()

raw_df["final_decision"].value_counts()

final_decision
REJECTED       7589
CONDITIONAL    2247
APPROVED        164
Name: count, dtype: int64